# Limpieza y unión del catálogo StreamView

Este notebook carga las fuentes oficiales, revisa su estructura, aplica limpieza verificable, homologa el esquema y exporta un catálogo integrado. No incluye EDA, gráficos ni análisis de negocio.

Regla de duplicados aplicada: se eliminan únicamente filas completamente idénticas. Los `show_id` repetidos entre Movies y TV Shows se conservan porque corresponden a contenidos distintos.

In [1]:
from pathlib import Path
import hashlib

import pandas as pd

pd.set_option('display.max_columns', None)

project_root = Path.cwd().resolve()
if not (project_root / 'data' / 'raw').exists():
    project_root = project_root.parent

movies_path = project_root / 'data' / 'raw' / 'netflix_movies_detailed_up_to_2025.csv'
tv_path = project_root / 'data' / 'raw' / 'netflix_tv_shows_detailed_up_to_2025.csv'
output_path = project_root / 'data' / 'processed' / 'catalogo_streamview.csv'

assert movies_path.exists(), f'No se encontró: {movies_path}'
assert tv_path.exists(), f'No se encontró: {tv_path}'

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

raw_hashes_before = {
    'movies': file_hash(movies_path),
    'tv_shows': file_hash(tv_path),
}


In [2]:
movies = pd.read_csv(movies_path)
tv_shows = pd.read_csv(tv_path)

revision = pd.DataFrame({
    'filas': [len(movies), len(tv_shows)],
    'columnas': [movies.shape[1], tv_shows.shape[1]],
    'duplicados_completos': [movies.duplicated().sum(), tv_shows.duplicated().sum()],
    'show_id_repetidos': [movies['show_id'].duplicated().sum(), tv_shows['show_id'].duplicated().sum()],
}, index=['Movies', 'TV Shows'])

display(revision)
display(pd.DataFrame({
    'nulos_movies': movies.isna().sum(),
    'nulos_tv_shows': tv_shows.isna().sum(),
}))

print('Columnas exclusivas de Movies:', sorted(set(movies.columns) - set(tv_shows.columns)))
print('Columnas exclusivas de TV Shows:', sorted(set(tv_shows.columns) - set(movies.columns)))

,filas,columnas,duplicados_completos,show_id_repetidos
Movies,16000,18,0,0
TV Shows,16000,16,0,9


,nulos_movies,nulos_tv_shows
budget,0,NaN
cast,204,1157.0
country,466,1797.0
date_added,0,0.0
description,132,3206.0
director,132,10965.0
duration,16000,0.0
genres,107,974.0
language,0,0.0
popularity,0,0.0


Columnas exclusivas de Movies: ['budget', 'revenue']
Columnas exclusivas de TV Shows: []


In [3]:
# Los vacíos de texto se normalizan a nulos; no se imputan valores.
def normalize_text_columns(dataframe):
    cleaned = dataframe.copy()
    for column in cleaned.select_dtypes(include='object').columns:
        cleaned[column] = cleaned[column].str.strip().replace('', pd.NA)
    return cleaned

movies_clean = normalize_text_columns(movies)
tv_shows_clean = normalize_text_columns(tv_shows)

# Solo se eliminan duplicados de filas completas, según DATA_RULES.md.
movies_clean = movies_clean.drop_duplicates().copy()
tv_shows_clean = tv_shows_clean.drop_duplicates().copy()

# Las variables exclusivas de Movies se mantienen como nulas para TV Shows.
for column in ['budget', 'revenue']:
    if column not in tv_shows_clean.columns:
        tv_shows_clean[column] = pd.NA

column_order = list(movies_clean.columns)
tv_shows_clean = tv_shows_clean.reindex(columns=column_order)

for dataframe in [movies_clean, tv_shows_clean]:
    dataframe['show_id'] = dataframe['show_id'].astype('string')
    dataframe['date_added'] = pd.to_datetime(dataframe['date_added'], errors='raise')
    dataframe['release_year'] = pd.to_numeric(dataframe['release_year'], errors='raise').astype('Int64')
    dataframe['vote_count'] = pd.to_numeric(dataframe['vote_count'], errors='raise').astype('Int64')
    for column in ['rating', 'popularity', 'vote_average', 'budget', 'revenue']:
        dataframe[column] = pd.to_numeric(dataframe[column], errors='raise')

print(f'Movies: {len(movies)} -> {len(movies_clean)} filas')
print(f'TV Shows: {len(tv_shows)} -> {len(tv_shows_clean)} filas')

Movies: 16000 -> 16000 filas
TV Shows: 16000 -> 16000 filas


/tmp/ipykernel_139035/755328641.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in cleaned.select_dtypes(include='object').columns:
/tmp/ipykernel_139035/755328641.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-d

In [4]:
catalogo = pd.concat([movies_clean, tv_shows_clean], ignore_index=True)

# Validaciones previas a la exportación.
assert set(catalogo['type'].dropna().unique()) == {'Movie', 'TV Show'}
assert not catalogo.duplicated().any(), 'Persisten filas completamente duplicadas.'
assert catalogo.loc[catalogo['type'].eq('TV Show'), ['budget', 'revenue']].isna().all().all()
assert raw_hashes_before == {
    'movies': file_hash(movies_path),
    'tv_shows': file_hash(tv_path),
}, 'Las fuentes originales fueron modificadas.'

catalogo['date_added'] = catalogo['date_added'].dt.strftime('%Y-%m-%d')
output_path.parent.mkdir(parents=True, exist_ok=True)
catalogo.to_csv(output_path, index=False)

print(f'Archivo exportado: {output_path}')
print(f'Filas: {len(catalogo):,}')
display(catalogo['type'].value_counts().rename_axis('type').to_frame('filas'))

Archivo exportado: /home/tomy/Downloads/Duoc/Visualizacion/visualizacion-de-datos-StreamView-Analytics/data/processed/catalogo_streamview.csv
Filas: 32,000


,filas
type,
Movie,16000
TV Show,16000


In [5]:
catalogo_verificado = pd.read_csv(output_path)

assert len(catalogo_verificado) == len(catalogo)
assert set(catalogo_verificado['type'].unique()) == {'Movie', 'TV Show'}
assert not catalogo_verificado.duplicated().any()

print('Validación final completada correctamente.')
display(catalogo_verificado.info())

Validación final completada correctamente.
<class 'pandas.DataFrame'>
RangeIndex: 32000 entries, 0 to 31999
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   show_id       32000 non-null  int64  
 1   type          32000 non-null  str    
 2   title         32000 non-null  str    
 3   director      20903 non-null  str    
 4   cast          30639 non-null  str    
 5   country       29737 non-null  str    
 6   date_added    32000 non-null  str    
 7   release_year  32000 non-null  int64  
 8   rating        32000 non-null  float64
 9   duration      16000 non-null  str    
 10  genres        30919 non-null  str    
 11  language      32000 non-null  str    
 12  description   28660 non-null  str    
 13  popularity    32000 non-null  float64
 14  vote_count    32000 non-null  int64  
 15  vote_average  32000 non-null  float64
 16  budget        16000 non-null  float64
 17  revenue       16000 non-null  float64

None